# Coqui TTS Training for Twi/Akan Languages

This notebook provides a complete pipeline for training Coqui TTS models on Twi data.

**Models Covered:**
- Tacotron2 (lightweight, good quality)
- VITS (end-to-end, fast inference)
- XTTS-v2 (voice cloning, multilingual)
- YourTTS (multi-speaker, zero-shot)

**Features:**
- Fine-tuning from pretrained models
- Custom Twi character support
- Voice cloning capabilities
- Multi-speaker training

## 1. Installation and Setup

In [ ]:
# Install Coqui TTS and dependencies
!pip install -q TTS==0.22.0
!pip install -q torch torchaudio
!pip install -q numpy scipy librosa
!pip install -q matplotlib tensorboard
!pip install -q phonemizer  # For G2P
!pip install -q wandb  # For experiment tracking (optional)

# For GPU training
!pip install -q accelerate

import os
import sys
import json
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import IPython.display as ipd
import matplotlib.pyplot as plt

# Coqui TTS imports
from TTS.api import TTS
from TTS.config import load_config
from TTS.tts.configs.tacotron2_config import Tacotron2Config
from TTS.tts.configs.vits_config import VitsConfig
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.models import setup_model
from TTS.trainer import Trainer, TrainerArgs
from TTS.utils.audio import AudioProcessor

print(f"PyTorch: {torch.__version__}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")
print(f"TTS Version: {TTS.__version__ if hasattr(TTS, '__version__') else 'Latest'}")

## 2. Data Preparation

In [ ]:
# Data paths
DATA_DIR = Path("/opt/kasaai/apps/voice-lab/data")
PROCESSED_DIR = DATA_DIR / "processed"
TTS_DATA_DIR = DATA_DIR / "tts_ready"
TTS_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Twi character set
TWI_CHARS = "aàáâãäbcdḍeèéêëɛfghiìíîïjklmnñoòóôõöɔpqrstuùúûüvwxyỳýz"
TWI_CHARS_UPPER = TWI_CHARS.upper()
PUNCTUATION = "!'(),-.:;? "
NUMBERS = "0123456789"

# Complete character set for Twi TTS
TWI_CHARSET = list(set(TWI_CHARS + TWI_CHARS_UPPER + PUNCTUATION + NUMBERS))
print(f"Twi character set size: {len(TWI_CHARSET)}")
print(f"Characters: {''.join(sorted(TWI_CHARSET))}")

In [ ]:
def prepare_coqui_dataset(csv_path: str, output_dir: Path, sample_rate: int = 22050):
    """Prepare dataset in Coqui TTS format.
    
    Coqui expects:
    - metadata.csv with format: audio_file|text|speaker_name
    - Audio files in wav format
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Load processed data
    df = pd.read_csv(csv_path)
    
    # Filter by duration (1-10 seconds is ideal for TTS)
    if 'duration' in df.columns:
        df = df[(df['duration'] >= 1.0) & (df['duration'] <= 10.0)]
    
    metadata = []
    audio_dir = output_dir / "wavs"
    audio_dir.mkdir(exist_ok=True)
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Preparing TTS data"):
        # Get audio path
        audio_path = row.get('audio_path', row.get('audio', ''))
        if not Path(audio_path).exists():
            continue
            
        # Copy/convert audio to wav
        import librosa
        import soundfile as sf
        
        audio_id = f"twi_{idx:06d}"
        output_audio = audio_dir / f"{audio_id}.wav"
        
        # Load and resample audio
        audio, sr = librosa.load(audio_path, sr=sample_rate)
        sf.write(output_audio, audio, sample_rate)
        
        # Get text (use processed if available)
        text = row.get('processed_text', row.get('text', ''))
        
        # Get speaker (default to single speaker)
        speaker = row.get('speaker', 'twi_speaker')
        
        # Add to metadata
        metadata.append(f"{output_audio.name}|{text}|{speaker}")
    
    # Save metadata
    metadata_path = output_dir / "metadata.csv"
    with open(metadata_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(metadata))
    
    print(f"✅ Prepared {len(metadata)} samples")
    print(f"📁 Data saved to: {output_dir}")
    print(f"📄 Metadata: {metadata_path}")
    
    return output_dir, len(metadata)

# Prepare dataset
if (DATA_DIR / "metadata" / "all_datasets_processed.csv").exists():
    dataset_dir, num_samples = prepare_coqui_dataset(
        str(DATA_DIR / "metadata" / "all_datasets_processed.csv"),
        TTS_DATA_DIR / "coqui_format"
    )
else:
    print("⚠️ No processed data found. Run data processing notebooks first.")

## 3. Tacotron2 Training

In [ ]:
def create_tacotron2_config(dataset_path: Path, output_path: Path):
    """Create Tacotron2 configuration for Twi."""
    
    config = Tacotron2Config(
        # Model
        model="tacotron2",
        
        # Audio
        audio={
            "sample_rate": 22050,
            "hop_length": 256,
            "win_length": 1024,
            "n_fft": 1024,
            "n_mels": 80,
            "mel_fmin": 0,
            "mel_fmax": 8000,
        },
        
        # Text
        characters={
            "characters_class": "TTS.tts.models.tacotron2.CharacterConfig",
            "pad": "_",
            "eos": "~",
            "bos": "^",
            "blank": None,
            "characters": ''.join(TWI_CHARSET),
            "punctuations": PUNCTUATION,
            "phonemes": None,
        },
        
        # Training
        batch_size=16 if torch.cuda.is_available() else 4,
        eval_batch_size=8,
        num_loader_workers=4,
        epochs=1000,
        lr=0.001,
        lr_scheduler="StepLR",
        lr_scheduler_params={"step_size": 50000, "gamma": 0.5},
        grad_clip=5.0,
        
        # Checkpointing
        save_step=5000,
        print_step=100,
        print_eval=True,
        mixed_precision=torch.cuda.is_available(),
        
        # Dataset
        output_path=str(output_path),
        datasets=[{
            "name": "twi_dataset",
            "path": str(dataset_path),
            "meta_file_train": "metadata.csv",
            "meta_file_val": None,
            "language": "tw",
        }],
        
        # Eval
        test_delay_epochs=100,
        test_sentences=[
            "Me ma wo akye.",
            "Wo ho te sɛn?",
            "Mepɛ sɛ mekasa Twi.",
            "Ɛyɛ sɛn? Me ho yɛ.",
        ],
    )
    
    # Save config
    config_path = output_path / "config.json"
    config.save_json(str(config_path))
    print(f"✅ Config saved to: {config_path}")
    
    return config

# Create Tacotron2 config
tacotron2_output = TTS_DATA_DIR / "models" / "tacotron2_twi"
tacotron2_output.mkdir(parents=True, exist_ok=True)

if 'dataset_dir' in locals():
    tacotron2_config = create_tacotron2_config(dataset_dir, tacotron2_output)

In [ ]:
# Training script for Tacotron2
tacotron2_train_script = """
# To train Tacotron2, run this in terminal:

python -m TTS.bin.train_tts \\
    --config_path {config_path} \\
    --coqpit.output_path {output_path} \\
    --coqpit.datasets.0.path {dataset_path} \\
    --coqpit.test_delay_epochs 100 \\
    --restore_path "" \\
    --best_path ""

# For fine-tuning from pretrained:
python -m TTS.bin.train_tts \\
    --config_path {config_path} \\
    --restore_path /path/to/pretrained/model.pth
""".format(
    config_path=tacotron2_output / "config.json",
    output_path=tacotron2_output,
    dataset_path=dataset_dir if 'dataset_dir' in locals() else "DATASET_PATH"
)

print("Tacotron2 Training Script:")
print("=" * 50)
print(tacotron2_train_script)

## 4. VITS Training (End-to-End)

In [ ]:
def create_vits_config(dataset_path: Path, output_path: Path):
    """Create VITS configuration for Twi.
    
    VITS is end-to-end, producing audio directly without vocoder.
    """
    
    config = VitsConfig(
        # Model
        model="vits",
        
        # Audio  
        audio={
            "sample_rate": 22050,
            "hop_length": 256,
            "win_length": 1024,
            "n_fft": 1024,
            "n_mels": 80,
            "mel_fmin": 0,
            "mel_fmax": None,
        },
        
        # Text
        use_phonemes=False,  # Can enable for better pronunciation
        text_cleaner="basic_cleaners",
        characters={
            "pad": "_",
            "eos": "~",
            "bos": "^",
            "blank": " ",
            "characters": ''.join(TWI_CHARSET),
            "punctuations": PUNCTUATION,
        },
        
        # Model architecture
        hidden_channels=192,
        inter_channels=192,
        filter_channels=768,
        n_layers=6,
        n_heads=2,
        kernel_size=3,
        dropout_p=0.1,
        
        # Training
        batch_size=16 if torch.cuda.is_available() else 2,
        eval_batch_size=8,
        epochs=10000,
        lr=0.0002,
        lr_scheduler="ExponentialLR",
        lr_scheduler_params={"gamma": 0.999875},
        grad_clip=1000,
        
        # Loss weights
        kl_loss_alpha=1.0,
        disc_loss_alpha=1.0,
        gen_loss_alpha=1.0,
        feat_loss_alpha=1.0,
        mel_loss_alpha=45.0,
        
        # Dataset
        output_path=str(output_path),
        datasets=[{
            "name": "twi_dataset",
            "path": str(dataset_path),
            "meta_file_train": "metadata.csv",
            "language": "tw",
        }],
    )
    
    # Save config
    config_path = output_path / "config.json"
    config.save_json(str(config_path))
    print(f"✅ VITS config saved to: {config_path}")
    
    return config

# Create VITS config
vits_output = TTS_DATA_DIR / "models" / "vits_twi"
vits_output.mkdir(parents=True, exist_ok=True)

if 'dataset_dir' in locals():
    vits_config = create_vits_config(dataset_dir, vits_output)

## 5. XTTS-v2 for Voice Cloning

In [ ]:
# XTTS setup for voice cloning
def setup_xtts_voice_cloning():
    """Setup XTTS for zero-shot voice cloning."""
    
    print("Setting up XTTS-v2 for voice cloning...")
    
    # Download XTTS model
    model_name = "tts_models/multilingual/multi-dataset/xtts_v2"
    
    try:
        tts = TTS(model_name, gpu=torch.cuda.is_available())
        print(f"✅ XTTS model loaded: {model_name}")
        
        # Test voice cloning
        test_text = "Me ma wo akye. This is a test of voice cloning."
        
        # You need a reference audio file
        reference_audio = DATA_DIR / "reference_voice.wav"
        
        if reference_audio.exists():
            output_path = TTS_DATA_DIR / "xtts_clone_test.wav"
            
            tts.tts_to_file(
                text=test_text,
                speaker_wav=str(reference_audio),
                language="en",  # XTTS doesn't have Twi, use English
                file_path=str(output_path)
            )
            
            print(f"✅ Voice cloning test saved to: {output_path}")
            return tts
        else:
            print("⚠️ No reference audio found for voice cloning")
            return tts
            
    except Exception as e:
        print(f"❌ Failed to load XTTS: {e}")
        return None

# Setup XTTS
xtts_model = setup_xtts_voice_cloning()

In [ ]:
def clone_voice_batch(reference_audios: list, texts: list, output_dir: Path):
    """Clone voices for multiple speakers."""
    
    if not xtts_model:
        print("❌ XTTS model not loaded")
        return
    
    output_dir.mkdir(parents=True, exist_ok=True)
    results = []
    
    for i, (ref_audio, text) in enumerate(zip(reference_audios, texts)):
        try:
            output_path = output_dir / f"cloned_{i:03d}.wav"
            
            xtts_model.tts_to_file(
                text=text,
                speaker_wav=ref_audio,
                language="en",
                file_path=str(output_path)
            )
            
            results.append({
                "reference": ref_audio,
                "text": text,
                "output": str(output_path),
                "status": "success"
            })
            
        except Exception as e:
            results.append({
                "reference": ref_audio,
                "text": text,
                "status": "failed",
                "error": str(e)
            })
    
    # Save results
    with open(output_dir / "cloning_results.json", "w") as f:
        json.dump(results, f, indent=2)
    
    success_count = sum(1 for r in results if r["status"] == "success")
    print(f"✅ Cloned {success_count}/{len(results)} voices")
    
    return results

## 6. Fine-tuning Pretrained Models

In [ ]:
def finetune_pretrained_model(model_name: str, dataset_path: Path, output_path: Path, epochs: int = 100):
    """Fine-tune a pretrained Coqui model on Twi data."""
    
    print(f"Fine-tuning {model_name} on Twi data...")
    
    # Load pretrained model
    tts = TTS(model_name, gpu=torch.cuda.is_available())
    
    # Get model config
    config = tts.config
    
    # Update config for Twi
    config.datasets = [{
        "name": "twi_dataset",
        "path": str(dataset_path),
        "meta_file_train": "metadata.csv",
        "language": "tw",
    }]
    
    config.output_path = str(output_path)
    config.epochs = epochs
    config.batch_size = 8 if torch.cuda.is_available() else 2
    
    # Add Twi characters
    if hasattr(config, 'characters'):
        config.characters['characters'] = ''.join(TWI_CHARSET)
    
    # Save updated config
    config_path = output_path / "config_finetuned.json"
    config.save_json(str(config_path))
    
    print(f"✅ Config saved to: {config_path}")
    print("\nTo fine-tune, run:")
    print(f"python -m TTS.bin.train_tts --config_path {config_path} --restore_path {model_name}")
    
    return config

# Example: Fine-tune English model on Twi
if 'dataset_dir' in locals():
    finetune_output = TTS_DATA_DIR / "models" / "finetuned_tacotron2"
    finetune_output.mkdir(parents=True, exist_ok=True)
    
    finetune_config = finetune_pretrained_model(
        "tts_models/en/ljspeech/tacotron2-DDC",
        dataset_dir,
        finetune_output
    )

## 7. Training Monitoring

In [ ]:
# TensorBoard monitoring
print("To monitor training with TensorBoard:")
print("="*50)
print("tensorboard --logdir=/opt/kasaai/apps/voice-lab/data/tts_ready/models")
print("\nThen open: http://localhost:6006")

# WandB setup (optional)
wandb_setup = """
# For Weights & Biases tracking:
import wandb

wandb.init(
    project="twi-tts",
    config={
        "model": "tacotron2",
        "dataset": "twi_combined",
        "samples": num_samples,
        "batch_size": 16,
        "learning_rate": 0.001,
    }
)
"""
print("\nFor experiment tracking with WandB:")
print("="*50)
print(wandb_setup)

## 8. Inference and Testing

In [ ]:
def test_trained_model(model_path: str, config_path: str, test_sentences: list):
    """Test a trained Coqui TTS model."""
    
    from TTS.utils.synthesizer import Synthesizer
    
    # Load model
    synthesizer = Synthesizer(
        tts_checkpoint=model_path,
        tts_config_path=config_path,
        use_cuda=torch.cuda.is_available()
    )
    
    outputs = []
    
    for i, text in enumerate(test_sentences):
        output_path = TTS_DATA_DIR / f"test_output_{i}.wav"
        
        # Synthesize
        wav = synthesizer.tts(text)
        
        # Save
        synthesizer.save_wav(wav, str(output_path))
        
        outputs.append({
            "text": text,
            "output": str(output_path)
        })
        
        print(f"✅ Synthesized: {text[:50]}...")
    
    return outputs

# Test sentences
test_sentences = [
    "Akwaaba! Me din de Kofi.",
    "Mepɛ sɛ mekasa Twi yie.",
    "Wo ho te sɛn? Me ho yɛ.",
    "Dr. Mensah will arrive at 3:30 PM.",
    "The price is GH₵500 for the items."
]

print("Test sentences ready for synthesis:")
for i, sent in enumerate(test_sentences, 1):
    print(f"{i}. {sent}")

## 9. Model Export and Optimization

In [ ]:
def export_for_production(model_path: str, config_path: str, output_dir: Path):
    """Export model for production deployment."""
    
    output_dir.mkdir(parents=True, exist_ok=True)
    
    export_formats = {
        "pytorch": ".pth",
        "onnx": ".onnx",
        "torchscript": ".pt"
    }
    
    print("Export options:")
    print("="*50)
    
    for format_name, ext in export_formats.items():
        export_path = output_dir / f"model{ext}"
        print(f"\n{format_name.upper()}:")
        
        if format_name == "onnx":
            print(f"python -m TTS.bin.export --model_path {model_path} --output_path {export_path} --format onnx")
        elif format_name == "torchscript":
            print(f"python -m TTS.bin.export --model_path {model_path} --output_path {export_path} --format torchscript")
        else:
            print(f"cp {model_path} {export_path}")
    
    # Quantization for mobile
    print("\nFor mobile deployment (quantization):")
    print("="*50)
    print("""
import torch
from torch.quantization import quantize_dynamic

model = torch.load(model_path)
quantized = quantize_dynamic(model, {torch.nn.Linear}, dtype=torch.qint8)
torch.save(quantized, 'model_quantized.pth')
    """)

# Export paths
export_dir = TTS_DATA_DIR / "exported_models"
export_for_production(
    "path/to/best_model.pth",
    "path/to/config.json",
    export_dir
)

## 10. Quick Start Commands

In [ ]:
print("COQUI TTS QUICK START COMMANDS")
print("="*60)

commands = {
    "List models": "tts --list_models",
    "Download model": "tts --model_name tts_models/en/ljspeech/tacotron2-DDC",
    "Synthesize": 'tts --text "Me ma wo akye" --model_name MODEL_NAME --out_path output.wav',
    "Voice clone": 'tts --text "Hello" --model_name tts_models/multilingual/multi-dataset/xtts_v2 --speaker_wav reference.wav --language en --out_path clone.wav',
    "Train": "python -m TTS.bin.train_tts --config_path config.json",
    "Continue training": "python -m TTS.bin.train_tts --config_path config.json --restore_path checkpoint.pth",
    "Convert checkpoint": "python -m TTS.bin.convert_checkpoint --model_path old.pth --output_path new.pth",
}

for name, cmd in commands.items():
    print(f"\n{name}:")
    print(f"  $ {cmd}")

print("\n" + "="*60)
print("✅ Coqui TTS setup complete!")

## Summary

This notebook provides:

1. **Data Preparation** - Convert to Coqui format
2. **Tacotron2** - Lightweight, good quality
3. **VITS** - End-to-end, fast inference
4. **XTTS-v2** - Voice cloning capabilities
5. **Fine-tuning** - Adapt pretrained models
6. **Training monitoring** - TensorBoard/WandB
7. **Testing** - Inference pipeline
8. **Export** - Production deployment

### Next Steps:
1. Prepare larger Twi dataset
2. Train models on GPU
3. Evaluate with native speakers
4. Deploy best model to production